# HEST-1k Signature-Derived Spatial State Fidelity

This notebook audits predicted-vs-measured local state recovery from the coverage95 biological signature scores. States are fitted on measured signature profiles; predicted signature profiles are assigned to the same state centroids.

In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src" / "histoomnist").exists() and (ROOT.parent / "src" / "histoomnist").exists():
    ROOT = ROOT.parent

OUT = ROOT / "results/hest1k_human_visium_expression/biological_signature_states"
summary = json.loads((OUT / "run_summary.json").read_text(encoding="utf-8"))
overall = pd.read_csv(OUT / "state_fidelity_overall.csv")
by_slide = pd.read_csv(OUT / "state_fidelity_by_slide.csv")
by_organ = pd.read_csv(OUT / "state_fidelity_by_organ.csv")
states = pd.read_csv(OUT / "state_annotations.csv")
confusion = pd.read_csv(OUT / "state_confusion_matrix.csv")
maps = pd.read_csv(OUT / "spatial_state_map_manifest.csv")

assert summary["score_kind"] == "rate"
assert summary["n_states"] == 6
assert summary["n_signatures"] == 10
assert summary["n_spots"] == int(overall.loc[0, "n_spots"])
assert maps["written"].all()
summary

{'signature_dir': 'results/hest1k_human_visium_expression/biological_signatures',
 'score_kind': 'rate',
 'n_spots': 113647,
 'n_signatures': 10,
 'n_states': 6,
 'n_fit_spots': 50000,
 'adjusted_rand': 0.28702141714147217,
 'normalized_mutual_info': 0.27645687523235285,
 'best_match_accuracy': 0.5850660378188601,
 'state_mapping_pred_to_true': {'0': 0,
  '1': 1,
  '2': 2,
  '3': 3,
  '4': 4,
  '5': 5},
 'outputs': {'state_fidelity_overall': 'results/hest1k_human_visium_expression/biological_signature_states/state_fidelity_overall.csv',
  'state_fidelity_by_slide': 'results/hest1k_human_visium_expression/biological_signature_states/state_fidelity_by_slide.csv',
  'state_fidelity_by_organ': 'results/hest1k_human_visium_expression/biological_signature_states/state_fidelity_by_organ.csv',
  'state_annotations': 'results/hest1k_human_visium_expression/biological_signature_states/state_annotations.csv',
  'state_confusion_matrix': 'results/hest1k_human_visium_expression/biological_signature

In [2]:
overall[["n_spots", "adjusted_rand", "normalized_mutual_info", "best_match_accuracy", "n_states", "n_signatures", "n_fit_spots"]]

,n_spots,adjusted_rand,normalized_mutual_info,best_match_accuracy,n_states,n_signatures,n_fit_spots
0,113647,0.287021,0.276457,0.585066,6,10,50000


In [3]:
state_view = states[["state", "n_true_spots", "n_pred_spots", "top_true_signatures", "top_true_z"]]
assert len(state_view) == summary["n_states"]
state_view

,state,n_true_spots,n_pred_spots,top_true_signatures,top_true_z
0,0,27029,30641,endothelial|hypoxia_stress|stromal_ecm,0.709|0.176|-0.036
1,1,43005,40012,t_cell|proliferation|interferon_response,-0.337|-0.420|-0.452
2,2,7919,14649,epithelial_tumor|proliferation|interferon_resp...,1.481|1.446|1.324
3,3,3436,136,pan_immune|proliferation|interferon_response,3.789|3.671|3.471
4,4,8828,3800,stromal_ecm|b_cell|endothelial,1.919|1.758|1.569
5,5,23430,24409,epithelial_tumor|myeloid|pan_immune,0.483|0.348|0.151


In [4]:
organ_view = by_organ.sort_values("adjusted_rand", ascending=False)
assert set(["Bowel", "Brain", "Skin"]).issubset(set(organ_view["organ"]))
organ_view

,organ,n_spots,adjusted_rand,normalized_mutual_info,best_match_accuracy
1,Brain,29221,0.418693,0.201689,0.781287
4,Skin,12604,0.371938,0.299076,0.692875
0,Bowel,42772,0.122085,0.163584,0.447840
2,Eye,1249,0.026210,0.028003,0.594876
3,Heart,27801,-0.009764,0.015457,0.540628


In [5]:
slide_view = by_slide.sort_values("adjusted_rand", ascending=False).head(12)
slide_view[["sample_id", "organ", "n_spots", "adjusted_rand", "normalized_mutual_info", "best_match_accuracy"]]

,sample_id,organ,n_spots,adjusted_rand,normalized_mutual_info,best_match_accuracy
40,NCBI656,Brain,4941,1.000000,1.000000,1.000000
43,TENX91,Bowel,5943,0.267882,0.358998,0.557463
45,ZEN44,Bowel,1048,0.243546,0.165548,0.608779
0,MEND35,Bowel,3273,0.224999,0.246908,0.553009
42,TENX90,Bowel,6042,0.164650,0.250185,0.452665
16,MISC33,Bowel,3385,0.162429,0.102966,0.643722
24,MISC73,Bowel,3885,0.124440,0.088881,0.641956
26,NCBI467,Skin,1851,0.120304,0.118178,0.532145
34,NCBI509,Skin,760,0.110429,0.048705,0.903947
5,MISC102,Heart,1164,0.098429,0.047275,0.958763


In [6]:
confusion

,true_state,0,1,2,3,4,5
0,0,16965,7323,636,0,217,1888
1,1,8755,29555,1141,0,27,3527
2,2,176,58,4609,11,427,2638
3,3,85,0,1887,121,191,1152
4,4,2246,93,1586,4,2468,2431
5,5,2414,2983,4790,0,470,12773


In [7]:
maps

,sample_id,path,written
0,MISC33,results/hest1k_human_visium_expression/biologi...,True
1,MISC46,results/hest1k_human_visium_expression/biologi...,True
2,MISC53,results/hest1k_human_visium_expression/biologi...,True
